# 01 — Generate reasoning traces

GPU. Run once per model; set `MODEL_NAME` in the config cell. Resumable: completed problems are skipped.

Writes `traces/{model}/{problem_id}.json` and `dataset_summary.json` under `DATA_ROOT`.

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# =============================================================================
# 01 — GENERATE REASONING TRACES
#
# Samples 300 problems from EleutherAI/hendrycks_math (train split; 200 at
# level 4-5 = "hard", 100 at level 3 = "medium"; random.Random(42)) and
# generates one greedy trace per problem per model (max 8,192 new tokens).
#
# Run once per model, changing MODEL_NAME in CELL 3:
#   deepseek-ai/DeepSeek-R1-Distill-Llama-8B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-14B
#   meta-llama/Llama-3.1-8B-Instruct           (non-reasoning control)
#
# The first run writes dataset_summary.json; later runs reload it so every
# model sees the same problems in the same order.
#
# OUTPUT: DATA_ROOT/traces/{model_short_name}/{problem_id}.json, run_log.json
#         DATA_ROOT/dataset_summary.json
#
# Note: the R1 chat template already ends with the opening <think> tag, so the
# generated text contains only "</think>" and `think_content` stays empty.
# Downstream notebooks therefore analyse `response_clean` (reasoning plus the
# final answer).
# =============================================================================

In [ ]:
# -----------------------------------------------------------------------------
# CELL 1 — Install dependencies
# -----------------------------------------------------------------------------
!pip install transformers accelerate datasets bitsandbytes -q

In [ ]:
# -----------------------------------------------------------------------------
# CELL 2 — HuggingFace login (run once per session)
# -----------------------------------------------------------------------------
# from huggingface_hub import login
# login()   # paste your HF token when prompted
#            # required for Llama-3.1-8B-Instruct (gated model)

In [ ]:
# -----------------------------------------------------------------------------
# CELL 3 — Imports and config
# -----------------------------------------------------------------------------
import os
import re
import json
import time
import random
import torch
from datetime import datetime
from pathlib import Path
from collections import defaultdict
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── MODEL SELECTION ───────────────────────────────────────────────────────────
# Change ONLY this between runs. Everything else is automatic.

#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# ── GENERATION CONFIG ─────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 8192    # hard cap ~5 min per problem; truncated traces dropped at analysis
DO_SAMPLE      = False   # greedy decoding — deterministic, reproducible

# ── DATASET CONFIG ────────────────────────────────────────────────────────────
N_HARD    = 200    # hendrycks level 4-5 (hard, AIME-equivalent difficulty)
N_MEDIUM  = 100    # hendrycks level 3   (medium, AMC-equivalent difficulty)
RANDOM_SEED = 42  # fixed for reproducibility across all model runs

# ── HENDRYCKS SUBJECT CONFIGS ─────────────────────────────────────────────────
HENDRYCKS_CONFIGS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

# ── QUANTIZATION ──────────────────────────────────────────────────────────────
USE_4BIT = "32B" in MODEL_NAME or "32b" in MODEL_NAME

# ── PATHS ─────────────────────────────────────────────────────────────────────
GDRIVE_BASE    = DATA_ROOT
MODEL_SHORT    = MODEL_NAME.split("/")[-1]
TRACE_DIR      = GDRIVE_BASE / "traces" / MODEL_SHORT
LOG_FILE       = TRACE_DIR / "run_log.json"
SUMMARY_FILE   = GDRIVE_BASE / "dataset_summary.json"

print(f"Model:       {MODEL_NAME}")
print(f"Short name:  {MODEL_SHORT}")
print(f"Output dir:  {TRACE_DIR}")
print(f"Use 4-bit:   {USE_4BIT}")
print(f"Max tokens:  {MAX_NEW_TOKENS}")
print(f"Target:      {N_HARD} hard + {N_MEDIUM} medium = {N_HARD + N_MEDIUM} problems")

In [ ]:
# -----------------------------------------------------------------------------
# CELL 4 — Mount Google Drive
# -----------------------------------------------------------------------------
TRACE_DIR.mkdir(parents=True, exist_ok=True)
GDRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Trace dir ready: {TRACE_DIR}")

In [ ]:
# -----------------------------------------------------------------------------
# CELL 5 — Load and sample problems from EleutherAI/hendrycks_math
# -----------------------------------------------------------------------------
def load_problems(n_hard: int, n_medium: int, seed: int) -> list[dict]:
    """
    Load from EleutherAI/hendrycks_math across all subject configs.
    Stratify by level:
        level 4-5 → "hard"    (N_HARD problems)
        level 3   → "medium"  (N_MEDIUM problems)
        level 1-2 → dropped   (too easy, insufficient revisions expected)

    Returns problems sorted: hard first, then medium.
    """
    print("Loading EleutherAI/hendrycks_math...")
    all_splits = []
    for config in HENDRYCKS_CONFIGS:
        try:
            ds = load_dataset(
                "EleutherAI/hendrycks_math",
                config,
                split="train",
                trust_remote_code=False
            )
            all_splits.append(ds)
            print(f"  {config:30s}: {len(ds)} problems")
        except Exception as e:
            print(f"  WARNING — could not load '{config}': {e}")

    if not all_splits:
        raise RuntimeError("Could not load any hendrycks configs. Check HF login.")

    full_ds = concatenate_datasets(all_splits)
    print(f"\nTotal loaded: {len(full_ds)} problems across all subjects")

    # Separate into difficulty pools
    hard_pool   = []
    medium_pool = []

    boxed_re = re.compile(r'\\boxed\{([^}]*)\}')

    for i, row in enumerate(full_ds):
        # Parse "Level N" → integer
        level_str = str(row.get("level", "Level 3"))
        try:
            level = int(re.search(r'\d', level_str).group())
        except (AttributeError, ValueError):
            level = 3

        # Skip levels 1-2
        if level < 3:
            continue

        # Extract answer from last \boxed{} in solution
        solution = row.get("solution", "")
        matches  = boxed_re.findall(solution)
        answer   = matches[-1].strip() if matches else ""

        entry = {
            "problem":  row["problem"],
            "answer":   answer,
            "solution": solution,       # keep for answer verification
            "source":   "EleutherAI/hendrycks_math",
            "subject":  row.get("type", "unknown"),
            "level":    level,
            "year":     "various",
        }

        if level >= 4:
            entry["problem_id"]  = f"hard_{i:05d}"
            entry["difficulty"]  = "hard"
            hard_pool.append(entry)
        else:  # level == 3
            entry["problem_id"]  = f"med_{i:05d}"
            entry["difficulty"]  = "medium"
            medium_pool.append(entry)

    print(f"\nPool sizes:")
    print(f"  hard   (level 4-5): {len(hard_pool)}")
    print(f"  medium (level 3):   {len(medium_pool)}")

    # Sample with fixed seed
    rng = random.Random(seed)

    def safe_sample(pool, n, label):
        if len(pool) < n:
            print(f"  WARNING: only {len(pool)} {label} problems available "
                  f"(wanted {n}). Using all.")
            return list(pool)
        return rng.sample(pool, n)

    hard_sample   = safe_sample(hard_pool,   n_hard,   "hard")
    medium_sample = safe_sample(medium_pool, n_medium, "medium")

    # Sort: hard first, then medium (for logical progress display)
    problems = hard_sample + medium_sample

    print(f"\nFinal dataset: {len(problems)} problems")
    print(f"  hard:   {len(hard_sample)}")
    print(f"  medium: {len(medium_sample)}")

    # Subject distribution (informational)
    subject_counts = defaultdict(int)
    for p in problems:
        subject_counts[p["subject"]] += 1
    print("\nSubject distribution:")
    for subj, cnt in sorted(subject_counts.items(), key=lambda x: -x[1]):
        print(f"  {subj:35s}: {cnt}")

    return problems

In [ ]:
# -----------------------------------------------------------------------------
# CELL 6 — Dataset summary (shared across all model runs)
# -----------------------------------------------------------------------------
def get_or_create_problems() -> list[dict]:
    """
    If dataset_summary.json already exists on Drive, load that exact problem
    order (ensures all 5 models see identical problems).
    Otherwise, sample fresh and save.
    """
    if SUMMARY_FILE.exists():
        print(f"Loading existing dataset summary from Drive...")
        with open(SUMMARY_FILE) as f:
            summary = json.load(f)

        # Rebuild full problem list from summary
        # (summary stores metadata only; we need to reload problem text)
        problems_fresh = load_problems(N_HARD, N_MEDIUM, RANDOM_SEED)
        id_to_problem  = {p["problem_id"]: p for p in problems_fresh}
        ordered_ids    = [p["problem_id"] for p in summary["problems"]]
        problems       = [id_to_problem[pid] for pid in ordered_ids
                          if pid in id_to_problem]

        print(f"Restored {len(problems)} problems in original order.")
        return problems

    else:
        print("No existing summary — sampling fresh dataset...")
        problems = load_problems(N_HARD, N_MEDIUM, RANDOM_SEED)

        # Save summary
        summary = {
            "created":      datetime.now().isoformat(),
            "n_problems":   len(problems),
            "n_hard":       sum(1 for p in problems if p["difficulty"] == "hard"),
            "n_medium":     sum(1 for p in problems if p["difficulty"] == "medium"),
            "max_new_tokens": MAX_NEW_TOKENS,
            "random_seed":  RANDOM_SEED,
            "problems": [
                {
                    "problem_id": p["problem_id"],
                    "difficulty": p["difficulty"],
                    "level":      p["level"],
                    "subject":    p["subject"],
                    "answer":     p["answer"],
                }
                for p in problems
            ]
        }
        with open(SUMMARY_FILE, "w") as f:
            json.dump(summary, f, indent=2)
        print(f"Dataset summary saved to: {SUMMARY_FILE}")
        return problems


problems = get_or_create_problems()

In [ ]:
# -----------------------------------------------------------------------------
# CELL 7 — Checkpoint log
# -----------------------------------------------------------------------------
def load_run_log() -> dict:
    if LOG_FILE.exists():
        with open(LOG_FILE) as f:
            log = json.load(f)
        n_done   = len(log["completed"])
        n_failed = len(log["failed"])
        print(f"Resuming. Completed: {n_done} | Failed: {n_failed} | "
              f"Remaining: {len(problems) - n_done - n_failed}")
    else:
        log = {
            "model":     MODEL_NAME,
            "started":   datetime.now().isoformat(),
            "completed": [],
            "failed":    [],
        }
        print(f"Fresh run. {len(problems)} problems to generate.")
    return log


def save_run_log(log: dict):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)


run_log   = load_run_log()
completed = set(run_log["completed"])
remaining = [p for p in problems if p["problem_id"] not in completed]
print(f"Problems remaining this session: {len(remaining)}")

In [ ]:
# -----------------------------------------------------------------------------
# CELL 8 — Load model
# -----------------------------------------------------------------------------
def load_model_and_tokenizer(model_name: str, use_4bit: bool):
    print(f"\nLoading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model ({'4-bit' if use_4bit else 'fp16'})...")
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )

    model.eval()
    print("Model loaded.")
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME, USE_4BIT)

In [ ]:
# -----------------------------------------------------------------------------
# CELL 9 — Prompt builder
# -----------------------------------------------------------------------------
def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])

    if problem["difficulty"] == "hard":
        instruction = (
            "Solve the following competition mathematics problem carefully. "
            "Show your reasoning step by step and verify your answer."
        )
    else:
        instruction = (
            "Solve the following mathematics competition problem. "
            "Show your working step by step."
        )

    if is_reasoning:
        messages = [{"role": "user",
                     "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system",
             "content": "You are an expert mathematician. Solve problems with "
                        "clear step-by-step reasoning."},
            {"role": "user",
             "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
# -----------------------------------------------------------------------------
# CELL 10 — Answer extraction
# -----------------------------------------------------------------------------
def extract_answer(text: str) -> str:
    # Last \boxed{} is most reliable
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    # Fallback: "the answer is X"
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"

In [ ]:
# -----------------------------------------------------------------------------
# CELL 11 — Generation function
# -----------------------------------------------------------------------------
@torch.no_grad()
def generate_trace(problem: dict) -> dict:
    prompt     = build_prompt(problem, tokenizer, MODEL_NAME)
    inputs     = tokenizer(prompt, return_tensors="pt",
                           truncation=False).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0

    new_tokens     = output_ids[0][prompt_len:]
    full_response  = tokenizer.decode(new_tokens, skip_special_tokens=False)
    response_clean = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Extract <think> block (R1/QwQ models)
    think_content = ""
    if "<think>" in full_response and "</think>" in full_response:
        start = full_response.index("<think>") + len("<think>")
        end   = full_response.index("</think>")
        think_content = full_response[start:end].strip()

    analysis_text = think_content if think_content else response_clean
    model_answer  = extract_answer(analysis_text)
    truncated     = len(new_tokens) >= MAX_NEW_TOKENS - 10

    return {
        # Identity
        "problem_id":        problem["problem_id"],
        "model":             MODEL_NAME,
        # Problem metadata
        "problem":           problem["problem"],
        "answer_gt":         problem["answer"],
        "difficulty":        problem["difficulty"],
        "level":             problem["level"],
        "subject":           problem["subject"],
        # Token counts
        "prompt_tokens":     prompt_len,
        "response_tokens":   len(new_tokens),
        # Text
        "full_response":     full_response,
        "response_clean":    response_clean,
        "think_content":     think_content,
        # Answer
        "model_answer":      model_answer,
        "correct":           model_answer.strip() == problem["answer"].strip(),
        # Metadata
        "generation_time_s": round(elapsed, 2),
        "timestamp":         datetime.now().isoformat(),
        "max_new_tokens":    MAX_NEW_TOKENS,
        "truncated":         truncated,
    }

In [ ]:
# -----------------------------------------------------------------------------
# CELL 12 — Main generation loop
# -----------------------------------------------------------------------------
def save_trace(trace: dict):
    out = TRACE_DIR / f"{trace['problem_id']}.json"
    with open(out, "w") as f:
        json.dump(trace, f, indent=2, ensure_ascii=False)


# Track per-difficulty progress
diff_done  = defaultdict(int)
diff_total = defaultdict(int)
for p in problems:
    diff_total[p["difficulty"]] += 1
for p in problems:
    if p["problem_id"] in completed:
        diff_done[p["difficulty"]] += 1

print(f"\nStarting generation: {len(remaining)} problems")
print(f"Model: {MODEL_NAME}")
print("=" * 65)

for i, problem in enumerate(remaining):
    pid  = problem["problem_id"]
    diff = problem["difficulty"]

    h_prog = f"{diff_done['hard']}/{diff_total['hard']}"
    m_prog = f"{diff_done['medium']}/{diff_total['medium']}"

    print(f"\n[{i+1}/{len(remaining)}] {pid}  "
          f"diff={diff}  hard:{h_prog}  med:{m_prog}")

    try:
        trace = generate_trace(problem)
        save_trace(trace)

        run_log["completed"].append(pid)
        save_run_log(run_log)
        diff_done[diff] += 1

        trunc_flag = " [TRUNCATED — will skip in analysis]" if trace["truncated"] else ""
        correct    = "✓" if trace["correct"] else "✗"
        print(f"  {correct} ans={trace['model_answer']} (gt={trace['answer_gt']}) "
              f"| {trace['response_tokens']} tok "
              f"| {trace['generation_time_s']}s{trunc_flag}")

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  FAILED: {e}")
        run_log["failed"].append({"problem_id": pid, "error": str(e)})
        save_run_log(run_log)

print("\n" + "=" * 65)
print(f"Session complete. Completed: {len(run_log['completed'])} | "
      f"Failed: {len(run_log['failed'])}")